# Symbol Table Generation for a C Program

This program reads a C program, identifies declared variables, creates a symbol table, and simulates memory allocation for each variable.

Import Required Libraries

In [ ]:
import re

Enter the C Program

In [ ]:
lines = [
    "int main()",
    "{",
    "    int a = 260;",
    '    char s[10] = "abcd";',
    "    int b = -1;",
    "    float x = 10.5;",
    "    return 0;",
    "}"
]

Define Data Type Sizes

In [ ]:
type_sizes = {
    "char": 1,
    "int": 4,
    "float": 4
}

start_address = 0x1000

Create the Symbol Table and Memory

In [ ]:
symbol_table = []
memory = {}

current_address = start_address
declared_symbols = set()

Function to Store Bytes in Memory

In [ ]:
def store_bytes(address, value, size):
    for i in range(size):
        memory[address + i] = (value >> (8 * i)) & 0xFF

Process Variable Declarations

In [ ]:
for line_no, line in enumerate(lines, start=1):

    # Remove comments
    line = re.sub(r'//.*', '', line)

    # Match int, float and char declarations
    declaration = re.search(
        r'\b(int|float|char)\s+(.+?);',
        line
    )

    if not declaration:
        continue

    data_type = declaration.group(1)
    variables = declaration.group(2)

    # Handle comma-separated declarations
    parts = re.split(r',\s*(?![^[]*\])', variables)

    for part in parts:

        part = part.strip()

        # Match array declaration
        array_match = re.match(
            r'([A-Za-z_]\w*)\s*\[\s*(\d+)\s*\]\s*(?:=\s*"([^"]*)")?',
            part
        )

        if array_match:
            name = array_match.group(1)
            array_size = int(array_match.group(2))
            string_value = array_match.group(3)

            if name in declared_symbols:
                continue

            declared_symbols.add(name)

            total_bytes = type_sizes[data_type] * array_size

            symbol_table.append({
                "Name": name,
                "Data Type": data_type,
                "Line Number": line_no,
                "Memory Location": current_address,
                "Bytes": total_bytes
            })

            # Store character array
            if data_type == "char" and string_value is not None:

                values = [ord(c) for c in string_value]

                # Add null terminator
                values.append(0)

                for i in range(array_size):
                    if i < len(values):
                        memory[current_address + i] = values[i]
                    else:
                        memory[current_address + i] = 0

            current_address += total_bytes

        else:
            # Normal variable
            normal_match = re.match(
                r'([A-Za-z_]\w*)\s*(?:=\s*(.*))?',
                part
            )

            if not normal_match:
                continue

            name = normal_match.group(1)
            value = normal_match.group(2)

            if name in declared_symbols:
                continue

            declared_symbols.add(name)

            size = type_sizes[data_type]

            symbol_table.append({
                "Name": name,
                "Data Type": data_type,
                "Line Number": line_no,
                "Memory Location": current_address,
                "Bytes": size
            })

            # Store initialized value
            if value is not None:

                value = value.strip()

                if data_type == "int":
                    number = int(value)
                    store_bytes(current_address, number & 0xFFFFFFFF, 4)

                elif data_type == "float":
                    import struct
                    byte_values = struct.pack("<f", float(value))

                    for i, byte in enumerate(byte_values):
                        memory[current_address + i] = byte

                elif data_type == "char":
                    if value.startswith("'") and value.endswith("'"):
                        memory[current_address] = ord(value[1])

            current_address += size

Display the Symbol Table

In [ ]:
print("SYMBOL TABLE")
print("-" * 75)

print(f"{'Name':<12}{'Data Type':<12}{'Line No.':<12}{'Memory':<15}{'Bytes':<8}")
print("-" * 75)

for symbol in symbol_table:
    print(
        f"{symbol['Name']:<12}"
        f"{symbol['Data Type']:<12}"
        f"{symbol['Line Number']:<12}"
        f"{hex(symbol['Memory Location']).upper():<15}"
        f"{symbol['Bytes']:<8}"
    )

SYMBOL TABLE
---------------------------------------------------------------------------
Name        Data Type   Line No.    Memory         Bytes   
---------------------------------------------------------------------------
a           int         3           0X1000         4       
s           char        4           0X1004         10      
b           int         5           0X100E         4       
x           float       6           0X1012         4       


Display Simulated Memory

In [ ]:
print("SIMULATED MEMORY")
print("-" * 30)
print(f"{'Address':<12}{'Content (Hex)':<15}")
print("-" * 30)

for address in sorted(memory):
    print(f"{address:04X}{'':<8}{memory[address]:02X}")

SIMULATED MEMORY
------------------------------
Address     Content (Hex)  
------------------------------
1000        04
1001        01
1002        00
1003        00
1004        61
1005        62
1006        63
1007        64
1008        00
1009        00
100A        00
100B        00
100C        00
100D        00
100E        FF
100F        FF
1010        FF
1011        FF
1012        00
1013        00
1014        28
1015        41


Display Memory by Symbol

In [ ]:
print("MEMORY ALLOCATION BY SYMBOL")
print("-" * 60)

for symbol in symbol_table:

    name = symbol["Name"]
    address = symbol["Memory Location"]
    size = symbol["Bytes"]

    print(f"\n{name} ({symbol['Data Type']})")
    print(f"Starting Address: {address:04X}")
    print(f"Bytes Used: {size}")

    for i in range(size):
        addr = address + i
        value = memory.get(addr, 0)
        print(f"  {addr:04X} -> {value:02X}")

MEMORY ALLOCATION BY SYMBOL
------------------------------------------------------------

a (int)
Starting Address: 1000
Bytes Used: 4
  1000 -> 04
  1001 -> 01
  1002 -> 00
  1003 -> 00

s (char)
Starting Address: 1004
Bytes Used: 10
  1004 -> 61
  1005 -> 62
  1006 -> 63
  1007 -> 64
  1008 -> 00
  1009 -> 00
  100A -> 00
  100B -> 00
  100C -> 00
  100D -> 00

b (int)
Starting Address: 100E
Bytes Used: 4
  100E -> FF
  100F -> FF
  1010 -> FF
  1011 -> FF

x (float)
Starting Address: 1012
Bytes Used: 4
  1012 -> 00
  1013 -> 00
  1014 -> 28
  1015 -> 41


Test with Another C Program

In [ ]:
lines = [
    "int main()",
    "{",
    "    int x = 100;",
    "    int y = 200;",
    "    char name[10] = \"Ria\";",
    "    float marks = 95.5;",
    "    return 0;",
    "}"
]

# Algorithm

1. Read the C program line by line.
2. Initialize the starting memory address as 1000H.
3. Identify variable declarations using regular expressions.
4. Extract the variable name, data type, and initialization value.
5. Check whether the symbol has already been declared.
6. Determine the number of bytes required based on its data type.
7. Assign the current memory address to the symbol.
8. Store the value in the simulated memory in hexadecimal form.
9. Update the current memory address according to the number of bytes used.
10. Store the symbol information in the symbol table.
11. Display the symbol table.
12. Display the simulated memory.